# Summit Healthcare Enterprise Data Warehouse
## Project Overview

This notebook outlines the steps to create the Summit Healthcare enterprise data warehouse (EDW) by connecting to PostgreSQL and executing the SQL schema to create a dimensional model (star schema) for pharmacy analytics. The resulting database serves as the foundation for synthetic data generation, SQL analytics, and business intelligence dashboards (Power BI) created in future stages of this project.

## Objectives
0. Project Overview
1. Import Libraries
2. Connect to PostgreSQL
3. Create (or connect to) the database
4. Read schema.sql
5. Execute schema.sql
6. Verify the tables exist
7. Close the connection

# Step 1. Import Python Libraries
1. psycopg2 --> connect Python to PostgreSQL
2. pathlib --> read the SQL schema file
3. pandas --> for future project phase, data manipulation

In [5]:
import psycopg2
import pandas as pd
from pathlib import Path

# Step 2. Connect to PostgreSQL
--Assumeds PostgreSQL has previously been downloaded and installed. 
--Creates a connection from Python to the PostgreSQL server.
--The connection will be used to execute the SQL schema and build the Summit Healthcare Enterprise Data Warehouse.

In [11]:
#import psycopg2

try:
    # Connect to PostgreSQL
    connection = psycopg2.connect(
        host="localhost",
        database="shedw",
        user="postgres",
        password="Woodscross25!",
        port="5432"
    )

    # Create a cursor (connection to database)
    cursor = connection.cursor()

    print("Successfully connected to SHEDW database!")

except Exception as e:
    print("Connection failed.")
    print(e)

Successfully connected to SHEDW database!


# Step 4. Read Schema SQL
--Notebook will open the sql/schema/schema.sql file and read it contents
--Maintain table definitions in one place

In [19]:
#read file
schema_path = Path("../sql/schema/schema.sql")

with open(schema_path, "r") as file:
    schema_sql = file.read()

print("schema.sql loaded successfully")
print(f"File path: {schema_path}")
print(f"Characters loaded: {len(schema_sql)}")

schema.sql loaded successfully
File path: ../sql/schema/schema.sql
Characters loaded: 3316


# Step 5. Execute the SQL
--Python sends everything in schema.sql to PostgreSQL which then creates all the tables.
--After this step, the database actually exists!

In [20]:
#execute file

try:
    cursor.execute(schema_sql)
    connection.commit()
    print("Schema executed successfully. Tables created.")

except Exception as e:
    connection.rollback()
    print("Schema execution failed.")
    print(e)

Schema executed successfully. Tables created.


# Step 6.1 Verify Table Creation/Print Tables
This section queries PostgreSQL to confirm that all tables were created successfully by querying information_schema.tables, a built-in PostgreSQL metadata table.

In [21]:
verify_tables_sql = """
SELECT table_schema,
       table_name
FROM information_schema.tables
WHERE table_schema = 'warehouse'
ORDER BY table_name;
"""

cursor.execute(verify_tables_sql)

tables = cursor.fetchall()

for table in tables:
    print(f"{table[0]}.{table[1]}")

warehouse.dim_department
warehouse.dim_medication
warehouse.dim_patient
warehouse.dim_provider
warehouse.fact_medication_dispense
warehouse.fact_medication_orders
warehouse.fact_medication_verification


# Step 6.2 Validate tables are empty
This section checks the number of rows in each warehouse table. Each table should contain zero rows.

In [22]:
for table in tables:
    schema_name = table[0]
    table_name = table[1]

    cursor.execute(f"SELECT COUNT(*) FROM {schema_name}.{table_name};")
    row_count = cursor.fetchone()[0]

    print(f"{schema_name}.{table_name}: {row_count} rows")

warehouse.dim_department: 0 rows
warehouse.dim_medication: 0 rows
warehouse.dim_patient: 0 rows
warehouse.dim_provider: 0 rows
warehouse.fact_medication_dispense: 0 rows
warehouse.fact_medication_orders: 0 rows
warehouse.fact_medication_verification: 0 rows


# Step 7. Close the database connection
Close the database cursor and ocnnection once build/validation steps are complete.

In [23]:
cursor.close()
connection.close()

print("Database connection closed.")

Database connection closed.


### Commit progress to Git.